In [1]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.optimizers import Adam

Make an expandable neural network model

In [6]:
def build_model(states: int, actions: int, hidden_layers=[24, 24]):
    """
    states - number of states 
    actions - number of actions
    """
    model = Sequential()
    
    # Flatten input layer (for handling state input shape)
    model.add(Flatten(input_shape=(1, states)))
    
    # Dynamically add hidden layers
    for neurons in hidden_layers:
        model.add(Dense(neurons, activation='relu'))
    
    # Output layer with softmax activation
    model.add(Dense(actions, activation='softmax'))
    
    return model

model = build_model(states=10, actions=4)
model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten_3 (Flatten)             │ (None, 10)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 24)             │           264 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 24)             │           600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 4)              │           100 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 964 (3.77 KB)

 Trainable params: 964 (3.77 KB)

 Non-trainable params: 0 (0.00 B)

Build an Agent with Keras

In [ ]:
from rl.agents import DQNAgent
from rl.policy import BoltzmannQPolicy
from rl.memory import SequentialMemory

In [ ]:
def build_agent(model, actions):
    policy = BoltzmannQPolicy()
    memory = SequentialMemory(limit=50000, window_length=1)
    dqn = DQNAgent(model=model, memory=memory, policy=policy,
                   nb_actions=actions, nb_steps_warmup=10, target_model_update=1e-2)
    return dqn


Use the agent to train the RL model

In [ ]:
dqn = build_agent(model, actions=4)
dqn.compile(Adam(lr=1e-3), metrics=['mae'])
dqn.fit(env, nb_steps=50000, visualize=False, verbose=1)

Chatty says that we can make the environment in the way that the gymnasium library does it so we can use all methods from them. 

Adam optimizer - adjusts the learning rate for each parameter in a neural network.

In the exploration, we would like to exploit all the information present in the estimated Q values produced by our network. The Boltzmann exploration does this. Instead of always taking a random or optimal action, this approach involves choosing an action with weighted probabilities. To accomplish this, it uses a softmax over the networks estimates of value for each action. In this case, the action that the agent estimates to be the optimal one is most likely (but not guaranteed) to be chosen. The biggest advantage over the e-greedy algorithm is that information about the likely value of the other actions can also be taken into consideration.

A metric is a function that is used to judge the performance of your model.
Metric functions are similar to loss functions, except that the results from evaluating a metric are not used when training the model. Note that you may use any loss function as a metric.